<a href="https://colab.research.google.com/github/zakateryila/Zaka-Teryila-Peter-/blob/main/Day_11_lab_cell_by_cell.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install vaderSentiment -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 6.3 MB/s eta 0:00:00


In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()
print(analyzer.polarity_scores("The app is good."))
{'neg': 0.0, 'neu': 0.508, 'pos': 0.492, 'compound': 0.4404}

{'neg': 0.0, 'neu': 0.508, 'pos': 0.492, 'compound': 0.4404}


{'neg': 0.0, 'neu': 0.508, 'pos': 0.492, 'compound': 0.4404}

In [ ]:
for text in ["The app is good.",
"The app is GOOD.",
"The app is good!!!",
"The app is very good.",
"The app is not good."]:
    score = analyzer.polarity_scores(text)["compound"]
    print(f"{score:+.4f} {text}")

+0.4404 The app is good.
+0.5622 The app is GOOD.
+0.5826 The app is good!!!
+0.4927 The app is very good.
-0.3412 The app is not good.


In [ ]:
for text in ["The app is not good.",
"The app is not good at all.",
"Wonderful. Third outage this week.",
"App dey crash anytime I wan transfer.",
"Abeg make una fix this app."]:
    score = analyzer.polarity_scores(text)["compound"]
    print(f"{score:+.4f} {text}")

-0.3412 The app is not good.
-0.3412 The app is not good at all.
+0.5719 Wonderful. Third outage this week.
-0.4019 App dey crash anytime I wan transfer.
+0.0000 Abeg make una fix this app.


In [ ]:
def vader_label(text, pos_cut=0.05, neg_cut=-0.05):
    """Turn VADER's compound score into one of our three labels."""
    score = analyzer.polarity_scores(text)["compound"]
    if score >= pos_cut:
        return "positive"
    if score <= neg_cut:
        return "negative"
    return "neutral"

print(vader_label("The app is not good at all."))
print(vader_label("Transfer was fast, I like am."))
print(vader_label("It works."))

negative
positive
neutral


In [ ]:
TEST_SET = [
("The app is fast and the transfer went through immediately.", "positive"),
("Transfer was fast, I like am.", "positive"),
("Best banking app I have used, no stress at all.", "positive"),
("Customer care called me back within 5 minutes. Impressed.", "positive"),
("Finally they fixed the login issue. Working well now.", "positive"),
("The transfer failed but support sorted it quickly.", "positive"),
("App dey crash anytime I wan transfer.", "negative"),
("They charged me twice and nobody is responding.", "negative"),
("The app is not good at all.", "negative"),
("Abeg make una fix this app.", "negative"),
("Wonderful. Third outage this week.", "negative"),
("I have been on hold for 40 minutes. Terrible service.", "negative"),
("Why is the OTP not coming? This is frustrating.", "negative"),
("Good app, but the fees are too high.", "negative"),
("I downloaded the app yesterday.", "neutral"),
("Does this app work on Android 14?", "neutral"),
("It works.", "neutral"),
("Please add dark mode in the next update.", "neutral"),
("The app size is 85MB.", "neutral"),
("I use this app for my business account.", "neutral"),
]
LABELS = ["positive", "neutral", "negative"]
print(f"{len(TEST_SET)} items")

20 items


In [ ]:
from collections import Counter
print(Counter(label for _, label in TEST_SET))
Counter({'negative': 8, 'positive': 6, 'neutral': 6})

Counter({'negative': 8, 'positive': 6, 'neutral': 6})


Counter({'negative': 8, 'positive': 6, 'neutral': 6})

In [ ]:
def always_positive(text):
    return "positive"

gold = [label for _, label in TEST_SET]
pred = [always_positive(text) for text, _ in TEST_SET]
correct = sum(g == p for g, p in zip(gold, pred))
print(f"Baseline accuracy: {correct}/{len(gold)} = {correct/len(gold):.0%}")

Baseline accuracy: 6/20 = 30%


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

def evaluate(predict_fn, name):
    """Score any labelling function against the fixed test set."""
    gold = [label for _, label in TEST_SET]
    pred = [predict_fn(text) for text, _ in TEST_SET]
    correct = sum(g == p for g, p in zip(gold, pred))
    print(f"\n===== {name} =====")
    print(f"Accuracy: {correct}/{len(gold)} = {correct/len(gold):.0%}\n")
    cm = confusion_matrix(gold, pred, labels=LABELS)
    print(f"{'true \\ said':14}" + "".join(f"{l:>10}" for l in LABELS))
    for lab, row in zip(LABELS, cm):
        print(f"{lab:14}" + "".join(f"{v:>10}" for v in row))
    print()
    print(classification_report(gold, pred, labels=LABELS,
                                digits=2, zero_division=0))
    return pred

evaluate(always_positive, "BASELINE - always positive")


===== BASELINE - always positive =====
Accuracy: 6/20 = 30%

true \ said     positive   neutral  negative
positive               6         0         0
neutral                6         0         0
negative               8         0         0

              precision    recall  f1-score   support

    positive       0.30      1.00      0.46         6
     neutral       0.00      0.00      0.00         6
    negative       0.00      0.00      0.00         8

    accuracy                           0.30        20
   macro avg       0.10      0.33      0.15        20
weighted avg       0.09      0.30      0.14        20



['positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive']

In [ ]:
evaluate(vader_label, "VADER - default thresholds")


===== VADER - default thresholds =====
Accuracy: 15/20 = 75%

true \ said     positive   neutral  negative
positive               5         1         0
neutral                1         5         0
negative               2         1         5

              precision    recall  f1-score   support

    positive       0.62      0.83      0.71         6
     neutral       0.71      0.83      0.77         6
    negative       1.00      0.62      0.77         8

    accuracy                           0.75        20
   macro avg       0.78      0.76      0.75        20
weighted avg       0.80      0.75      0.75        20



['neutral',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'negative',
 'negative',
 'negative',
 'neutral',
 'positive',
 'negative',
 'negative',
 'positive',
 'neutral',
 'neutral',
 'neutral',
 'positive',
 'neutral',
 'neutral']